# model_experiment_PatchTST

PatchTST (Nie et al. 2023, *"A Time Series is Worth 64 Words"*) via the `neuralforecast` library. Two ideas define the architecture:
1. **Patching** — the input window is split into patches (like ViT tokens), so attention runs over ~13 patch tokens instead of 104 raw timesteps;
2. **Channel independence** — one shared Transformer, each of the 3331 series is forecast from its own history alone (no cross-series attention).

RevIN (reversible instance normalization) handles the 100x scale differences between series. Loss is MAE (matches the L1 nature of WMAE); the 5x holiday weighting cannot be injected into the library loss per-timestep — an honest limitation vs our LightGBM/DLinear setups, documented for the report.

**Kaggle settings: Accelerator = GPU T4, Internet = On.** MLflow experiment: **PatchTST_Training**.

In [1]:
# Kaggle bootstrap — attach competition data + the three MLFLOW secrets first.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow neuralforecast")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

import torch
print("GPU available:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.7 which is incompatible.
Cloning into 'walmart-sales-forecasting'...


GPU available: True


In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pandas as pd
import mlflow

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow
from src.postprocess import naive_lag52, apply_christmas_shift, blend_holiday_naive
from src.preprocessing import BLEND_HOLIDAY_WEEKS

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
setup_mlflow("PatchTST_Training")

# Pin training to ONE device: on multi-GPU Kaggle machines (T4 x2) Lightning
# otherwise launches DDP via fork, which cannot re-init CUDA in a notebook.
GPU_KW = ({"accelerator": "gpu", "devices": 1} if torch.cuda.is_available()
          else {"accelerator": "cpu"})


def to_nf(df):
    # neuralforecast long format: unique_id, ds, y
    out = df[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    out["unique_id"] = out["Store"].astype(str) + "_" + out["Dept"].astype(str)
    return out.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["unique_id", "ds", "y"]]


def from_nf(fc, model_col):
    out = fc.reset_index() if "unique_id" not in fc.columns else fc.copy()
    sd = out["unique_id"].str.split("_", expand=True).astype(int)
    out["Store"], out["Dept"] = sd[0], sd[1]
    return out.rename(columns={"ds": "Date", model_col: "pred"})[["Store", "Dept", "Date", "pred"]]

MLflow -> https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow | experiment: PatchTST_Training


## Run 1 — Preprocessing decisions

In [3]:
with mlflow.start_run(run_name="PatchTST_Preprocessing"):
    mlflow.log_params({
        "format": "long (unique_id=Store_Dept, ds, y), no exogenous features (channel-independent)",
        "min_series_len": "input handled by library windowing; series too short -> naive fallback at predict",
        "normalization": "RevIN (per-instance, reversible)",
        "loss": "MAE (per-timestep holiday weighting not supported by library loss — documented limitation)",
        "horizon": 39,
    })

🏃 View run PatchTST_Preprocessing at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3/runs/fa8ef2bfb1d342ceaa3d635b79a3c407
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3


## Run 2 — Fold-1 evaluation (patch_len ablation)

Fold-1 train is 91 weeks, so input_size=52 (one full year of context, the longest that leaves any training windows).

In [4]:
def eval_fold1(patch_len, stride, max_steps=1500, input_size=52, seed=42):
    tr, va = split_fold(train, 1)
    nf_train = to_nf(tr)
    model = PatchTST(h=39, input_size=input_size,
                     patch_len=patch_len, stride=stride,
                     hidden_size=128, n_heads=4, dropout=0.2,
                     revin=True, loss=MAE(),
                     max_steps=max_steps, batch_size=64,
                     windows_batch_size=512, random_seed=seed,
                     start_padding_enabled=True, **GPU_KW)
    nf = NeuralForecast(models=[model], freq="W-FRI")
    nf.fit(nf_train)
    fc = from_nf(nf.predict(), "PatchTST")
    m = va.merge(fc, on=["Store", "Dept", "Date"], how="left")
    coverage = m["pred"].notna().mean()
    m["pred"] = m["pred"].fillna(pd.Series(naive_lag52(tr, va), index=m.index))
    dep_med = tr.groupby("Dept")["Weekly_Sales"].median()
    m["pred"] = m["pred"].fillna(m["Dept"].map(dep_med)).fillna(0)
    rep = wmae_report(m["Weekly_Sales"], m["pred"], m["IsHoliday"])
    return rep, coverage, m, tr

results = {}
for patch_len, stride in ((16, 8), (8, 4)):
    rep, cov, m1, tr1 = eval_fold1(patch_len, stride)
    results[(patch_len, stride)] = rep["wmae"]
    with mlflow.start_run(run_name=f"PatchTST_CV_patch{patch_len}"):
        mlflow.log_params({"patch_len": patch_len, "stride": stride, "input_size": 52,
                           "max_steps": 1500, "fold": 1})
        mlflow.log_metrics({**rep, "coverage": cov})
    print(f"patch_len={patch_len}: coverage {cov:.3f}, {rep}")

BEST_PATCH, BEST_STRIDE = min(results, key=results.get)
print("best:", BEST_PATCH, BEST_STRIDE)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 430 K  | train
------------------------------------------------------------------
430 K     Trainable params
3         Non-trainable params
430 K     Total params
1.722     Total estimated model params size (MB)
93        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1500` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

🏃 View run PatchTST_CV_patch8 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3/runs/09e5ce6349ff453eb9ee0c337fad7dd6
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3
patch_len=8: coverage 0.992, {'wmae': 2904.642043834261, 'mae_holiday': 4091.3804810614565, 'mae_nonholiday': 2403.3683313362294}
best: 16 8


## Run 3 — Holiday blend (no Christmas; motivation in the LightGBM notebook)

In [5]:
rep1, _, m1, tr1 = eval_fold1(BEST_PATCH, BEST_STRIDE)
blend_scores = {}
for w in (0.0, 0.5, 0.75):
    blended = blend_holiday_naive(m1[["Store", "Dept", "Date", "pred"]], tr1,
                                  weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    rep = wmae_report(m1["Weekly_Sales"], blended["pred"], m1["IsHoliday"])
    blend_scores[w] = rep["wmae"]
    with mlflow.start_run(run_name=f"PatchTST_Blend_noXmas_w{w}"):
        mlflow.log_params({"patch_len": BEST_PATCH, "blend_weight": w, "fold": 1})
        mlflow.log_metrics(rep)
    print(f"w={w}: {rep}")
BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 430 K  | train
------------------------------------------------------------------
430 K     Trainable params
3         Non-trainable params
430 K     Total params
1.722     Total estimated model params size (MB)
93        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1500` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

🏃 View run PatchTST_Blend_noXmas_w0.0 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3/runs/b6e81c3602b54f55a35dcb3b91130829
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3
w=0.0: {'wmae': 2764.5769189786997, 'mae_holiday': 3932.518674751355, 'mae_nonholiday': 2271.2428518142897}
🏃 View run PatchTST_Blend_noXmas_w0.5 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3/runs/8cdf9f8a079c4dddbe35c0039372dec8
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3
w=0.5: {'wmae': 2420.768964666861, 'mae_holiday': 2774.7639599500285, 'mae_nonholiday': 2271.2428518142897}
🏃 View run PatchTST_Blend_noXmas_w0.75 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3/runs/c55def5f384849e2b53aa312b4d67a58
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.ml

## Run 4 — Final: train on ALL data (input 104 = two seasonal cycles), predict test

In [6]:
final_model = PatchTST(h=39, input_size=104,
                       patch_len=BEST_PATCH, stride=BEST_STRIDE,
                       hidden_size=128, n_heads=4, dropout=0.2,
                       revin=True, loss=MAE(),
                       max_steps=3000, batch_size=64,
                       windows_batch_size=512, random_seed=42,
                       start_padding_enabled=True, **GPU_KW)
nf = NeuralForecast(models=[final_model], freq="W-FRI")
nf.fit(to_nf(train))

fc = from_nf(nf.predict(), "PatchTST")
sub = test[["Store", "Dept", "Date"]].merge(fc, on=["Store", "Dept", "Date"], how="left")
covered = sub["pred"].notna().mean()
sub["pred"] = sub["pred"].fillna(pd.Series(naive_lag52(train, test), index=sub.index))
dep_med = train.groupby("Dept")["Weekly_Sales"].median()
sub["pred"] = sub["pred"].fillna(sub["Dept"].map(dep_med)).fillna(0)

sub = apply_christmas_shift(sub, pred_col="pred")
if BLEND_W > 0:
    sub = blend_holiday_naive(sub, train, weight=BLEND_W, holiday_dates=BLEND_HOLIDAY_WEEKS)

with mlflow.start_run(run_name="PatchTST_Final"):
    mlflow.log_params({"input_size": 104, "patch_len": BEST_PATCH, "stride": BEST_STRIDE,
                       "max_steps": 3000, "blend_weight": BLEND_W,
                       "post": "christmas_shift + noXmas_blend"})
    mlflow.log_metrics({"fold1_wmae": results[(BEST_PATCH, BEST_STRIDE)],
                        "test_coverage": covered})
    nf.save(path="patchtst_model", overwrite=True)
    mlflow.log_artifacts("patchtst_model", artifact_path="model")

make_submission(sub, "pred", "submission_patchtst.csv")
print(f"wrote submission_patchtst.csv (coverage {covered:.3f}, blend w={BLEND_W})")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 466 K  | train
------------------------------------------------------------------
466 K     Trainable params
3         Non-trainable params
466 K     Total params
1.865     Total estimated model params size (MB)
93        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=3000` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

🏃 View run PatchTST_Final at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3/runs/2cfcf4dd7b9f4190bbc28c490508ff5c
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/3
wrote submission_patchtst.csv (coverage 0.996, blend w=0.75)
